# Cell-Akamai companion 03 · Prefill, decode and prefix reuse
**Local Ollama on this host — NOT an Akamai GPU run, NOT vLLM.** Original workshop notebook `03_prefill_decode_kv_cache/03_prefill_decode_kv_cache.ipynb` is unchanged; this notebook consumes the same *question* on a different backend. Results are never pooled with Akamai runs. Status: **NOT EXECUTED** until the executed copy exists.

Server-reported `prompt_eval_*` / `eval_*` from Ollama give prefill and decode rates. **Client TTFC is first streamed chunk, not server TTFT.** No KV-utilization metric exists on Ollama — that Akamai/vLLM measurement is `NOT_AVAILABLE` here. Run with nothing else using this Ollama server.

In [ ]:
import sys, json, pathlib
sys.path[:0] = ["lane_b/companion", "lane_b/harness"]   # run from the repo root
import cakm, pandas as pd
OUT = pathlib.Path("runs/companion"); OUT.mkdir(parents=True, exist_ok=True)
MODELS = ["hf.co/LiquidAI/LFM2.5-350M-GGUF:Q4_K_M", "hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF:Q4_K_M"]  # small models; edit to taste
HOST = cakm.host_info(); print(json.dumps(HOST, indent=1))

In [ ]:
SENT = "The quick measurement of cached attention state reveals how prefill cost grows with context length. "
KS = [10, 40, 160, 240]; REPEATS = 3; NP = 32; OPT = {"num_ctx": 4096}
rows = []
for m in MODELS:
    w = cakm.generate_stream(m, SENT * 4, num_predict=8, options=OPT); w.update(phase="warmup", model=m); rows.append(w)
    for k in KS:
        for r in range(REPEATS):   # unique salt at the START defeats prefix reuse between repeats
            p = f"Document R{k}-{r}. " + SENT * k
            x = cakm.generate_stream(m, p, num_predict=NP, options=OPT); x.update(phase="prefill_sweep", k=k, repeat=r); rows.append(x)
    p = "Document REUSE. " + SENT * 160     # prefix reuse: identical prompt twice
    for j in range(2):
        x = cakm.generate_stream(m, p, num_predict=NP, options=OPT); x.update(phase="prefix_reuse", k=160, repeat=j); rows.append(x)
len(rows)

In [ ]:
rec = cakm.write_receipt(OUT/"03", "prefill_decode", rows, {"module": "03", "host": HOST, "models": MODELS, "num_ctx": 4096, "num_predict": NP})
df = pd.DataFrame(rows); ok = df[(df.status == "OBSERVED") & (df.phase != "warmup")].copy()
ok["prefill_tok_s"] = ok.prompt_eval_count / ok.prompt_eval_duration_s
ok["decode_tok_s"] = ok.eval_count / ok.eval_duration_s
ok["tpot_s_server_mean"] = ok.eval_duration_s / ok.eval_count
print("failed:", int((df.status == "FAILED").sum()), "| receipt", rec["jsonl_sha256"])
ok[ok.phase == "prefill_sweep"].groupby(["model", "k"]).agg(prompt_tokens=("prompt_eval_count", "median"), ttfc_s=("ttfc_s", "median"),
    prefill_tok_s=("prefill_tok_s", "median"), decode_tok_s=("decode_tok_s", "median"), tpot_s=("tpot_s_server_mean", "median")).reset_index()

In [ ]:
ok[ok.phase == "prefix_reuse"][["model", "repeat", "prompt_eval_count", "prompt_eval_duration_s", "ttfc_s", "load_duration_s"]]

**Reading it:** medians over 3 repeats; with so few repeats no variance claim is made. If the second prefix-reuse call reports a much smaller `prompt_eval_count`, that is Ollama's reported cache reuse — an observation about this server, not about vLLM prefix caching.